In [ ]:
import os

# Aktuelles Arbeitsverzeichnis nehmen
script_dir = os.getcwd()

# Eine Ebene hoch
main_dir = os.path.abspath(os.path.join(script_dir, ".."))
os.chdir(main_dir)
import sys
if main_dir not in sys.path:
    sys.path.insert(0, main_dir)

print(f"📁 Arbeitsverzeichnis gesetzt auf: {os.getcwd()}")

In [ ]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import psycopg2

from functions import read_db_credentials, connect_to_db, load_json_data_from_db_as_json, save_df_to_db, save_df_to_db

In [ ]:
# def read_db_credentials(path="data/config.txt"):
#     creds = {}
#     with open(path, "r") as f:
#         for line in f:
#             key, value = line.strip().split("=")
#             creds[key] = value
#     return creds

# def connect_to_db(creds):
#     return psycopg2.connect(
#         host=creds["host"],
#         port=creds["port"],
#         dbname=creds["database"],
#         user=creds["user"],
#         password=creds["password"]
#     )


In [ ]:
with open("data/cur_user_selected.txt", "r", encoding="utf-8") as f:
    user = int(f.read().strip())



# def load_json_data_from_db_as_json(user, source ):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)

#     query = f"SELECT raw_json FROM fact_raw_data WHERE data_source = '{source}' AND user_number = {user} ;"
    
#     cursor = conn.cursor()
#     cursor.execute(query)
#     result = cursor.fetchone()
#     conn.close()
#     #return (result)
#     # # Parsen der JSON-Inhalte aus der 'data'-Spalte
#     parsed_data = result[0] if result else {}


#     # # Rückgabe als JSON-String (optional indent für Lesbarkeit)
#     return parsed_data



strava_data = load_json_data_from_db_as_json(user, "strava")


In [ ]:
# Top-Level Keys anzeigen
data = strava_data
import json

def print_json_structure(data, indent=0):
    spacer = "  " * indent
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{spacer}\"{key}\": ", end="")
            if isinstance(value, (dict, list)):
                print()
                print_json_structure(value, indent + 1)
            else:
                print(type(value).__name__)
    elif isinstance(data, list):
        print(f"{spacer}[")
        if data:
            print_json_structure(data[0], indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}<empty>")
        print(f"{spacer}]")
    else:
        print(f"{spacer}{type(data).__name__}")

# JSON-Datei laden

# Struktur ausgeben
print_json_structure(data)


# ACTIVITIES

In [ ]:
activities = strava_data.get("activities", [])

#TODO english version


strava_activity_types = [
    "Lauf", "Traillauf", "Spaziergang", "Wandern", "Virtueller Lauf",
    "Radfahrt", "Mountainbike-Fahrt", "Gravel-Fahrt", "E-Bike-Radfahrt", "E-Mountainbike-Fahrt",
    "Velomobil", "Virtuelle Radfahrt", "Kanu", "Kajakfahrt", "Kitesurfen", "Rudern",
    "Stand-up-Paddling", "Surfen", "Schwimmen", "Windsurfen", "Eislaufen", "Ski Alpin",
    "Tourenski", "Ski Nordisch", "Snowboarden", "Schneeschuhwanderung", "Handbike-Fahrt",
    "Inlineskaten", "Klettern", "Rollski", "Golf", "Skateboarden", "Fußball", "Rollstuhlfahrt",
    "Badminton", "Tennis", "Pickleball", "Crossfit", "Crosstrainer", "Stufen-Stepper",
    "Gewichtstraining", "Yoga", "Training", "HIIT", "Pilates", "Tischtennis", "Squash",
    "Racquetball"
]

# Mapping basierend auf typischer Kategorisierung
def classify_user_activity_type(activity):
    cardio = ["lauf", "trail", "spazier", "wandern", "virtuel", "rad", "gravel", "velo", "bike",
              "kajak", "kanu", "rudern", "paddling", "surf", "schwimm", "ski", "snow", "skate", "inlineskate", "eis"]
    strength = ["crossfit", "gewicht", "hiit","training"]
    flexibility = ["yoga", "pilates"]
    wellness = ["golf", "spazier", "wandern", "tischtennis", "badminton", "tennis", "squash", "pickle", "racquet"]

    a = activity.lower()
    if any(k in a for k in strength):
        return "Strength"
    elif any(k in a for k in flexibility):
        return "Flexibility"
    elif any(k in a for k in wellness):
        return "Wellness"
    elif any(k in a for k in cardio):
        return "Cardio"
    else:
        return "Other"

extracted_rows = []
for act in activities:
    art = act.get("Aktivitätsart", "Unbekannt")
    dauer = act.get("Verstrichene Zeit", 0)
    distanz = act.get("Distanz.1")/1000
    timestamp = act.get("Aktivitätsdatum")
    dauer_min = round(dauer / 60, 2)
    extracted_rows.append({
        "activity": art,
        "timestamp": pd.to_datetime(timestamp, format="%d.%m.%Y, %H:%M:%S"),
        "duration": dauer_min,
        "distance": distanz
    })

df_activities = pd.DataFrame(extracted_rows)
df_activities ["type"] = df_activities ["activity"].apply(classify_user_activity_type)
print(df_activities)



In [ ]:
df_steps = pd.read_pickle("tmp/activites.pkl")
df_activites_tot = pd.concat([df_activities,df_steps])
print(df_activites_tot)

In [ ]:
import pandas as pd

# Beispiel-Input (nur Cardio, aber Code funktioniert allgemein)

# Entferne doppelte Spalten
df_activites_score = df_activites_tot.loc[:, ~df_activites_tot.columns.duplicated()]

# Score-Spalte initialisieren
df_activites_score["score"] = 0.0

# Cardio-Score berechnen
cardio_mask = df_activites_score["type"] == "Cardio"
df_activites_score.loc[cardio_mask, "score"] = (
    df_activites_score.loc[cardio_mask, "duration"] * 0.6 +
    df_activites_score.loc[cardio_mask, "distance"] * 0.4
)

# Strength-Score (fix)
strength_mask = df_activites_score["type"] == "Strength"
df_activites_score.loc[strength_mask, "score"] = 10

# Wellness-Score (Dauer-basiert)
wellness_mask = df_activites_score["type"] == "Wellness"
df_activites_score.loc[wellness_mask, "score"] = df_activites_score.loc[wellness_mask, "duration"]

# Flexibility-Score (Dauer-basiert)
flex_mask = df_activites_score["type"] == "Flexibility"
df_activites_score.loc[flex_mask, "score"] = df_activites_score.loc[flex_mask, "duration"]

# Zusammenfassung
score_summary = df_activites_score.groupby("type")["score"].sum()
score_parts = score_summary / score_summary.sum()

# Ausgabe
print("\nScore Summary by Type:")
print(score_summary)

print("\nNormalized Score Proportions:")
print(score_parts)


# MOTIVATION

In [ ]:
def get_motivation(data):
    score = {
        "Performance": 0,
        "Social Recognition": 0,
        "Gamification": 0
    }

    # Ziele oder Performancesmotivierte Aktivitätsmetriken
    if len(data.get("goals", [])) > 0:
        score["Performance"] += 2

    for a in data.get("activities", []):
        if not isinstance(a, dict):
            continue
        if a.get("Relative Performance") is not None:
            score["Performance"] += 1
        if a.get("Gefühlte Anstrengung") is not None:
            score["Performance"] += 1

    # Soziales Verhalten
    social_settings = data.get("social_settings", [{}])
    if social_settings and isinstance(social_settings[0], dict):
        for v in social_settings[0].values():
            if isinstance(v, str) and "aktiviert" in v.lower():
                score["Social Recognition"] += 1

    # Kudos aus reactions
    kudo_count = sum(
        1 for r in data.get("reactions", [])
        if isinstance(r, dict) and r.get("Reaktionstyp", "").lower() == "kudos"
    )
    score["Social Recognition"] += kudo_count * 1

    # Herausforderungen
    if len(data.get("global_challenges", [])) > 0:
        score["Gamification"] += 2
    if len(data.get("group_challenges", [])) > 0:
        score["Gamification"] += 2

    # Bewertung: stärkste Motivation
    if all(v == 0 for v in score.values()):
        return "Unklar"
    
    # Max-Wert und Rückgabe der zugehörigen Motivation
    dominant = max(score, key=score.get)
    return {"dominant": dominant, **score}


print(get_motivation(strava_data))

# EVENT DOG

# STRAVA SOCIAL SCORE

In [ ]:
from datetime import datetime

def get_social_interaction_score(data):
    followers = len(data.get("followers", []))
    following = len(data.get("following", []))
    clubs = len(data.get("clubs", [])) + len(data.get("memberships", []))

    # Zeitspanne bestimmen (von erster bis letzter Aktivität)
    dates = [
        datetime.strptime(a["Aktivitätsdatum"], "%d.%m.%Y, %H:%M:%S")
        for a in data.get("activities", [])
        if isinstance(a, dict) and "Aktivitätsdatum" in a
    ]

    if not dates:
        active_weeks = 1  # Annahme, falls keine Daten
    else:
        delta_days = (max(dates) - min(dates)).days
        active_weeks = max(delta_days // 7, 1)

    # Pro-Woche-Berechnung
    comments_total = len(data.get("comments", []))
    reactions_total = len(data.get("reactions", []))
    comments_per_week = comments_total / active_weeks
    reactions_per_week = reactions_total / active_weeks

    # Normierung: z. B. 5 pro Woche = Maximum (1.0)
    norm_comments = min(comments_per_week / 1, 1)
    norm_reactions = min(reactions_per_week / 3, 1)

    # Feste Grenzwerte für andere Metriken
    score = (
        min(followers / 20, 1) +
        min(following / 20, 1) +
        min(clubs / 5, 1) +
        norm_comments +
        norm_reactions
    ) / 5

    return round(score, 2)



In [ ]:

def parse_activity_date(date_str):
    try:
        return datetime.strptime(date_str, "%d.%m.%Y, %H:%M:%S")
    except Exception:
        return None

def attends_events(data):
    keywords = ["rennen", "marathon", "event", "race", "triathlon", "5k", "10k"]

    # Events zählen
    event_dates = []
    for e in data.get("events", []):
        date = e.get("Datum") or e.get("date")  # je nach Export
        dt = parse_activity_date(date) if date else None
        if dt:
            event_dates.append(dt.year)

    # Aktivitäten nach Name durchsuchen
    for a in data.get("activities", []):
        if not isinstance(a, dict):
            continue
        name = a.get("Name der Aktivität", "").lower()
        if any(kw in name for kw in keywords):
            dt = parse_activity_date(a.get("Aktivitätsdatum", ""))
            if dt:
                event_dates.append(dt.year)

    # Jahresweise zählen
    from collections import Counter
    year_counts = Counter(event_dates)

    # Prüfen, ob in irgendeinem Jahr mehr als 2 vorkommen
    return any(count > 2 for count in year_counts.values())


In [ ]:
activity_counts = df_activites_tot["activity"].value_counts().head(4).reset_index()
print(df_activites_tot)
activity_counts.columns = ["activity", "count"]
df_activites_tot["timestamp"] = pd.to_datetime(df_activites_tot["timestamp"], errors="coerce")

def safe_activity_label(df, idx):
    if idx < len(df):
        row = df.iloc[idx]
        return f"{row['activity']}\n{row['count']}"
    else:
        return "NA"


In [ ]:
import numpy as np
import pandas as pd

# Nur einmalig sicherstellen, dass keine doppelten Spalten vorhanden sind
df_activites_tot = df_activites_tot.loc[:, ~df_activites_tot.columns.duplicated()]

# Datum vereinheitlichen (nur Datumsteil, ohne Uhrzeit)
df_activites_tot["timestamp"] = pd.to_datetime(df_activites_tot["timestamp"]).dt.normalize()


print(df_activites_tot["timestamp"])

# ➤ Strength-Aktivitäten pro Tag
strength_pw_mean = round((
    df_activites_tot[df_activites_tot["type"] == "Strength"]
    .groupby(pd.Grouper(key="timestamp", freq="W"))  # gruppiert nach Woche
    .size()  # zählt Einträge pro Woche
    .mean()  # Mittelwert der wöchentlichen Counts
),1)

# Falls Ergebnis NaN ist (z. B. keine Strength-Aktivitäten), dann 0
# Schritt 1: Doppelte Spalten sicher entfernen
df_activites_tot = df_activites_tot.loc[:, ~df_activites_tot.columns.duplicated()]

# Schritt 2: Timestamp korrekt parsen
df_activites_tot["timestamp"] = pd.to_datetime(df_activites_tot["timestamp"])

# Schritt 3: Hilfsspalte für Kalenderwoche (Periodenobjekt)
df_activites_tot["week"] = df_activites_tot["timestamp"].dt.to_period("W")

# Schritt 4: Cardio pro Woche summieren und Mittelwert berechnen
fl_cardio_min_pw = (
    df_activites_tot[df_activites_tot["type"] == "Cardio"]
    .groupby("week")["duration"]
    .sum()
    .mean()
)
pa_t1 = safe_activity_label(activity_counts, 0)
pa_t2 = safe_activity_label(activity_counts, 1)
pa_t3 = safe_activity_label(activity_counts, 2)
pa_t4 = safe_activity_label(activity_counts, 3)

# ➤ Score-Verteilung pro Aktivitätstyp
fintess_type_endurence = score_parts.get("Cardio", 0)
fitness_type_strength = score_parts.get("Strength", 0)
fintess_type_flexibility = score_parts.get("Flexibility", 0)  # korrigiert Schreibfehler
fintess_type_relaation = score_parts.get("Wellness", 0)       # korrigiert: 'relaation' war vermutlich Tippfehler

# ➤ Motivationstyp, Social Score, Eventteilnahme aus externen Funktionen
motivation_type = get_motivation(strava_data)["dominant"]
sc_strava_social_score = get_social_interaction_score(strava_data)
sc_strava_event_dog = attends_events(strava_data)


In [ ]:

creds = read_db_credentials()
conn = connect_to_db(creds)
cur = conn.cursor()

sql = """
INSERT INTO dim_strava (
    user_number,
    fl_workouts_pw,
    fl_cardio_min_pw,
    pa_t1,
    pa_t2,
    pa_t3,
    pa_t4,
    fitness_type_endurence,
    fitness_type_strength,
    fitness_type_flexibility,
    fitness_type_relaation,
    motivation_type,
    sc_strava_social_score,
    sc_strava_event_dog
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

values = (
    int(user),
    float(strength_pw_mean),
    float(fl_cardio_min_pw),
    str(pa_t1),
    str(pa_t2),
    str(pa_t3),
    str(pa_t4),
    float(fintess_type_endurence),
    float(fitness_type_strength),
    float(fintess_type_flexibility),
    float(fintess_type_relaation),
    str(motivation_type),
    float(sc_strava_social_score),
    bool(sc_strava_event_dog)
)

cur.execute(sql, values)
conn.commit()
cur.close()
conn.close()

print("✅ Daten erfolgreich in dim_strava gespeichert.")
